# DiagnoTech Model 2 · Notebook 03 — Final Evaluation, Analysis & Deployment

**Protocol.**
1. Select the experiment: the top-ranked eligible run from notebook 02, or one you name.
2. Tune per-class decision thresholds on the **validation** split only.
3. Evaluate **once** on the **test** split. The test split is loaded in this notebook and nowhere else.
4. Export deployment artifacts and check that the deployment pipeline reproduces the training model's probabilities.

**Multi-label metrics.** Each image can carry several findings, so the reported measures are per class. Macro AUROC is threshold-free. Label accuracy counts each label decision separately. Exact-match accuracy is strict, because a sample counts only if all eight labels are correct. Accuracy alone is not used for judgement.

## 1. Setup

In [ ]:
import json
import os
import shutil
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

CANDIDATE_CODE_ROOTS = [
    os.environ.get("DIAGNOTECH_CODE_ROOT"),
    "/content/DiagnoTech_Model2",
    str(Path.cwd()),
    str(Path.cwd().parent),
]
code_roots = [Path(c) for c in CANDIDATE_CODE_ROOTS if c and (Path(c) / "src" / "diagnotech").is_dir()]
if not code_roots:
    raise FileNotFoundError("Project code not found. Unzip DiagnoTech_Model2_Professional.zip to /content first.")
CODE_ROOT = code_roots[0]
sys.path.insert(0, str(CODE_ROOT / "src"))
sys.path.insert(0, str(CODE_ROOT / "deployment"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import multilabel_confusion_matrix, roc_curve

from diagnotech.checkpointing import load_checkpoint  # noqa: E402
from diagnotech.config import in_colab, load_context, load_json, save_json  # noqa: E402
from diagnotech.data_manifest import load_manifest, resolve_image_paths  # noqa: E402
from diagnotech.metrics import (  # noqa: E402
    binarize,
    exact_match_accuracy,
    hamming_accuracy,
    macro_auroc,
    macro_f1,
    per_class_report,
    rank_experiments,
    tune_thresholds,
)
from diagnotech.model import build_densenet121  # noqa: E402
from diagnotech.preprocessing import ChestXrayDataset, build_loader, build_transforms, load_rgb_image  # noqa: E402
from diagnotech.training import collect_predictions  # noqa: E402

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")

ctx = load_context(CODE_ROOT)
cfg = ctx.project
CLASSES = list(ctx.classes)
SEED = cfg["split"]["seed"]
RAW_DIR = Path(os.environ.get("DIAGNOTECH_RAW_DIR", cfg["source"]["colab_raw_data_dir"]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
plt.rcParams.update({"figure.dpi": 110})
print("Device:", device, "| storage:", ctx.storage.root)

## 2. Select the experiment
By default the top-ranked eligible experiment is chosen. Set `SELECTED_EXPERIMENT` to override, for example to report a simpler configuration with similar performance.

In [ ]:
results = pd.read_csv(ctx.storage.experiments / "experiment_results.csv")
ranking = rank_experiments(
    results,
    baseline_id=cfg["selection"]["baseline_experiment"],
    max_class_auroc_drop=cfg["selection"]["max_class_auroc_drop_vs_baseline"],
)
display(ranking[["rank", "experiment_id", "eligible", "val_macro_auroc", "val_macro_f1_tuned", "val_loss"]].round(4))

SELECTED_EXPERIMENT = None  # None -> top-ranked; or a string such as "E6"
SELECTED = SELECTED_EXPERIMENT or str(ranking.iloc[0]["experiment_id"])
best_path = ctx.storage.checkpoints / SELECTED / "best_model.pth"
if not best_path.exists():
    raise FileNotFoundError(f"No best checkpoint for {SELECTED}. Run notebook 02 for it first.")
print(f"Selected {SELECTED}: {ctx.experiments[SELECTED]['description']}")

## 3. Load the best checkpoint
The checkpoint is checked against the project class mapping and the preprocessing configuration, so inference uses exactly the transforms used in training.

In [ ]:
ckpt = load_checkpoint(best_path, map_location="cpu")
if list(ckpt["class_mapping"]) != CLASSES:
    raise ValueError("Checkpoint class mapping differs from the project. Refusing to evaluate.")
if ckpt["preprocessing"] != cfg["preprocessing"]:
    raise ValueError("Checkpoint preprocessing differs from project_config.json. Refusing to evaluate.")

model = build_densenet121(len(CLASSES), dropout=float(ckpt["dropout"]), pretrained=False)
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()
print(f"Loaded best epoch {ckpt['best_epoch']} | best validation loss {ckpt['best_val_loss']:.4f}")

eval_tf = build_transforms(cfg["preprocessing"], augment=False)


def make_loader(manifest: pd.DataFrame):
    return build_loader(
        ChestXrayDataset(manifest, CLASSES, eval_tf), batch_size=32, shuffle=False, sampler=None,
        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda", seed=SEED,
    )


manifests = {
    split: resolve_image_paths(load_manifest(ctx.storage.data / f"manifest_{split}.csv"), RAW_DIR)
    for split in ("val", "test")
}

## 4. Tune thresholds on validation (never on test)
Each class gets the threshold that maximises its F1 on the validation split. The thresholds are then fixed and applied unchanged to the test split.

In [ ]:
_, val_true, val_prob = collect_predictions(model, make_loader(manifests["val"]), device, use_amp=False)
THRESHOLDS = tune_thresholds(val_true, val_prob)
display(pd.DataFrame({"class": CLASSES, "threshold": THRESHOLDS.round(3)}).set_index("class").T)

## 5. Final evaluation on the untouched test split
The test split is used here for the first and only time. Predictions are collected in manifest order, so every row can be traced back to its image.

In [ ]:
_, test_true, test_prob = collect_predictions(model, make_loader(manifests["test"]), device, use_amp=False)
test_pred = binarize(test_prob, THRESHOLDS)
print(f"Test images: {len(test_true):,}")

tuned_report = per_class_report(test_true, test_prob, CLASSES, thresholds=THRESHOLDS)
default_report = per_class_report(test_true, test_prob, CLASSES, thresholds=0.5)

summary = {
    "experiment_id": SELECTED,
    "test_images": int(len(test_true)),
    "macro_auroc": macro_auroc(test_true, test_prob),
    "macro_average_precision": float(np.nanmean(tuned_report["average_precision"].to_numpy(dtype=float))),
    "macro_precision_tuned": float(tuned_report["precision"].mean()),
    "macro_recall_tuned": float(tuned_report["recall"].mean()),
    "macro_f1_tuned": macro_f1(test_true, test_pred),
    "macro_f1_at_0_5": macro_f1(test_true, binarize(test_prob, 0.5)),
    "label_accuracy_tuned": hamming_accuracy(test_true, test_pred),
    "exact_match_accuracy_tuned": exact_match_accuracy(test_true, test_pred),
    "min_class_auroc": float(np.nanmin(tuned_report["auroc"].to_numpy(dtype=float))),
    "thresholds": {c: float(t) for c, t in zip(CLASSES, THRESHOLDS)},
}
pd.Series({k: v for k, v in summary.items() if k != "thresholds"}, name="test").to_frame().round(4)

In [ ]:
display(tuned_report.round(4))
tuned_report.to_csv(ctx.storage.evaluation / "test_per_class_report_tuned.csv", index=False)
default_report.to_csv(ctx.storage.evaluation / "test_per_class_report_threshold_0_5.csv", index=False)

## 6. Confusion matrices and ROC curves

In [ ]:
cms = multilabel_confusion_matrix(test_true, test_pred)  # shape (K, 2, 2): [[tn, fp], [fn, tp]]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for ax, cls, cm in zip(axes.ravel(), CLASSES, cms):
    ax.imshow(cm, cmap="Blues")
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, f"{value:,}", ha="center", va="center", color="black")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["pred 0", "pred 1"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["true 0", "true 1"])
    ax.set_title(cls)
fig.suptitle(f"Per-class confusion matrices on the test split · {SELECTED} · thresholds from validation")
fig.tight_layout()
fig.savefig(ctx.storage.evaluation / "confusion_matrices_test.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for k, cls in enumerate(CLASSES):
    if test_true[:, k].min() == test_true[:, k].max():
        continue
    fpr, tpr, _ = roc_curve(test_true[:, k], test_prob[:, k])
    auc_value = tuned_report.set_index("class").loc[cls, "auroc"]
    ax.plot(fpr, tpr, label=f"{cls} (AUROC {auc_value:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("ROC curves, test split")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ctx.storage.evaluation / "roc_curves_test.png", bbox_inches="tight")
plt.show()

## 7. Per-class performance chart and error analysis
Precision, recall and F1 per class. Then the most frequent error patterns: which true label set was confused with which predicted one. Use this table to write the error analysis in `reports/`, and compare false negatives against Cardiomegaly and the other minority classes.

In [ ]:
perf = tuned_report.set_index("class")[["precision", "recall", "f1"]]
ax = perf.plot(kind="bar", figsize=(12, 4.8), rot=30)
ax.set_ylim(0, 1)
ax.set_title("Per-class precision, recall and F1 (test, thresholds from validation)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ctx.storage.evaluation / "per_class_performance_test.png", bbox_inches="tight")
plt.show()


def label_set(row: np.ndarray) -> tuple[str, ...]:
    return tuple(cls for cls, flag in zip(CLASSES, row) if flag)


error_patterns: Counter = Counter()
for true_row, pred_row in zip(test_true, test_pred):
    if not np.array_equal(true_row, pred_row):
        error_patterns[(label_set(true_row), label_set(pred_row))] += 1

error_table = pd.DataFrame(
    [{"true_labels": " | ".join(t) or "(none)", "predicted_labels": " | ".join(p) or "(none)", "count": n}
     for (t, p), n in error_patterns.most_common(15)]
)
print(f"Images with at least one wrong label decision: {sum(error_patterns.values()):,} of {len(test_true):,}")
display(error_table)
error_table.to_csv(ctx.storage.evaluation / "top_error_patterns_test.csv", index=False)

## 8. Save the evaluation summary
The summary is saved to Drive. Write your interpretation of the error analysis into `reports/` after reviewing the numbers above.

In [ ]:
save_json(summary, ctx.storage.evaluation / "final_evaluation_summary.json")
print(json.dumps({k: v for k, v in summary.items() if k != "thresholds"}, indent=2))

## 9. Deployment artifacts
Four artifacts are written to `deployment/`: `best_model.pth`, `class_mapping.json`, `model_config.json` (thresholds, preprocessing, dropout and version) and the inference code (`model_loader.py`, `inference.py`, which ship with the zip). The thresholds are the validation-derived values from section 4.

In [ ]:
DEPLOY_DIR = ctx.storage.deployment
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_path, DEPLOY_DIR / "best_model.pth")
save_json(load_json(CODE_ROOT / "config" / "class_mapping.json"), DEPLOY_DIR / "class_mapping.json")

model_version = f"{SELECTED}-{datetime.now(timezone.utc):%Y%m%d%H%M}"
model_config = {
    "model_version": model_version,
    "architecture": "densenet121",
    "dropout": float(ckpt["dropout"]),
    "classes": CLASSES,
    "thresholds": summary["thresholds"],
    "threshold_source": "validation split, maximum F1 per class",
    "preprocessing": cfg["preprocessing"],
    "weights_file": "best_model.pth",
    "experiment_id": SELECTED,
    "output": "independent sigmoid per class (multi-label)",
    "test_macro_auroc": summary["macro_auroc"],
}
save_json(model_config, DEPLOY_DIR / "model_config.json")
print("Deployment artifacts written to", DEPLOY_DIR)
print(sorted(p.name for p in DEPLOY_DIR.iterdir()))

## 10. Verify the deployment pipeline
The deployed predictor loads the same weights through `model_loader` and `inference`. For a sample of test images, its probabilities must match the training model's probabilities. A mismatch here would mean the backend returns predictions that differ from what was evaluated, and the notebook stops.

In [ ]:
from inference import DiagnoTechPredictor  # noqa: E402

predictor = DiagnoTechPredictor.from_directory(DEPLOY_DIR, device=device)
sample_paths = manifests["test"]["image_path"].head(8).tolist()
deployed = predictor.predict_batch(sample_paths)
deployed_probs = np.array([[r["probabilities"][c] for c in CLASSES] for r in deployed])

with torch.no_grad():
    batch = torch.stack([eval_tf(load_rgb_image(p)) for p in sample_paths]).to(device)
    direct_probs = torch.sigmoid(model(batch)).cpu().numpy()

max_abs_diff = float(np.max(np.abs(deployed_probs - direct_probs)))
print(f"Maximum |deployed - training model| probability difference over {len(sample_paths)} images: {max_abs_diff:.2e}")
assert max_abs_diff < 1e-4, "Deployment pipeline disagrees with the training model. Do not deploy."
print("Deployment pipeline verified against the training model.")

example = predictor.predict(sample_paths[0])
print(json.dumps({k: example[k] for k in ("predicted_labels", "top_3", "model_version")}, indent=2))

## 11. Limitations and integration notes

**Limitations.**
* Labels in the NIH ChestX-ray14 family were extracted automatically from radiology reports, and that process introduces label noise. Reported metrics measure agreement with those labels, not with independent clinical truth.
* Minority classes have few positive test images. Their per-class estimates are wide, and the warnings from notebook 01 apply.
* Exact image duplicates were detected by byte hash. Near-duplicates, such as re-compressed copies, are not detected.
* ONNX export is not included in this release, and no ONNX claim is made.
* This is a research prototype. It is not a validated medical device and must not be used for clinical decisions without independent regulatory and clinical validation.

**Backend integration.** Copy the `deployment/` folder, with its inference code, into the backend service. Call it as follows:

```python
from inference import DiagnoTechPredictor
predictor = DiagnoTechPredictor.from_directory("deployment/", device="cpu")
result = predictor.predict(image_bytes)   # bytes, a file path, or a PIL image
# result["predicted_labels"], result["top_3"], result["probabilities"], result["disclaimer"]
```

Load the predictor once at service start-up and reuse it across requests. The model is not thread-safe for concurrent training, but inference calls are read-only.